### Inference Pipeline


### 1. Set Kernel Env


In [ ]:
# %%bash

# PKGs=$(poetry env info --path)/lib/python3.9/site-packages
# echo $PKGs
# echo $PYTHONPATH
# export PYTHONPATH=$PYTHONPATH:$PKGs


In [ ]:
# %pip install -e /home/gorelova_i_v/projects/cvm_churn-from-dac_binary-class_churn-dac


In [ ]:
%load_ext autoreload
%load_ext dotenv
%dotenv
%autoreload 2


### 2. State and Connections


In [ ]:
try:
    spark.stop()
except NameError:
    pass

import logging
from datetime import datetime
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd

from cvm_model.io import State
import cvm_model.sql as sql
import cvm_model.utils as utils
from cvm_model.inference.preprocess import _prepare_dataset, _validate_dataset
from cvm_model.parameters import (
    aud_table,
    fav_omni_features_table,
    features,
    features_for_outliers,
    inference_data_stat_suffix,
    input_suffix,
    model_predictions_suffix,
    model_type,
    preperiod_months,
    project_name,
    score,
    target,
    template,
)

logging.basicConfig(level=logging.INFO)
pd.set_option('display.max_columns', None)

state = State.from_env()
engine = state.credentials.loyalty_gp.sa_engine
s3 = state.credentials.cvm_s3
session = state.spark.session

import os
os.environ.update(state.credentials.mlflow.environment)
if state.credentials.mlflow.tracking_uri:
    mlflow.set_tracking_uri(state.credentials.mlflow.tracking_uri)

print('MLflow URI:', mlflow.get_tracking_uri())


### 3. Set Event Timestamps


In [ ]:
event_timestamp = datetime(2026, 6, 1)
train_event_timestamp = datetime(2026, 6, 1)

event_date = event_timestamp.date().isoformat()
train_event_date = train_event_timestamp.date().isoformat()

print('Inference event date:', event_date)
print('Train event date:', train_event_date)


### 4. Load Inference Dataset


In [ ]:
input_prefix = state.settings.preprocess_prefix(event_timestamp) / input_suffix / 'inference'
input_bucket = input_prefix.split('//')[1].split('/')[0]
input_prefix = '/'.join(input_prefix.split('//')[1].split('/')[1:])
input_path = template.format(bucket=input_bucket, prefix=input_prefix)

print('Input path:', input_path)

df = session.read.parquet(input_path).toPandas()

assert len(df) > 0, 'Inference dataset is empty'
assert df['contact_id'].nunique() == len(df), 'Inference dataset contains duplicate contact_id values'
missing_features = set(features) - set(df.columns)
assert not missing_features, f'Inference dataset is missing features: {missing_features}'

print('Inference dataset shape:', df.shape)
display(df.head())


### 5. Data Quality Checks


In [ ]:
quality_report = pd.DataFrame({
    'metric': ['rows', 'unique_contact_id', 'features', 'missing_features'],
    'value': [len(df), df['contact_id'].nunique(), len(features), len(missing_features)],
})

display(quality_report)

if 'segment' in df.columns:
    display(
        df.groupby('segment', dropna=False)
        .agg(rows=('contact_id', 'size'))
        .assign(share=lambda x: x['rows'] / x['rows'].sum())
        .sort_values('rows', ascending=False)
    )


### 6. Load Model from MLflow


In [ ]:
client = mlflow.MlflowClient()
model_name = f'{project_name}_{model_type}'
versions = client.search_model_versions(
    f"name = '{model_name}'",
    order_by=['version_number DESC'],
)
assert len(versions) > 0, f'Registered model was not found: {model_name}'

matching_versions = [
    version
    for version in versions
    if version.tags.get('event_timestamp') == train_event_date
]
assert matching_versions, f'Model version for train_event_timestamp={train_event_date} was not found'

model_version = matching_versions[0]
print('Model:', model_version.name)
print('Version:', model_version.version)
print('Train event timestamp:', model_version.tags.get('event_timestamp'))

model = mlflow.sklearn.load_model(f'models:/{model_version.name}/{model_version.version}')


### 7. Score Dataset


In [ ]:
df[score] = model.predict_proba(df[features])[:, 1]

print('Prediction rows:', len(df))
print('Score min:', df[score].min())
print('Score mean:', df[score].mean())
print('Score max:', df[score].max())

display(df[['contact_id', 'segment', score]].head())


### 8. Save Predictions


In [ ]:
predictions_prefix = state.settings.s3_permanent_prefix / model_predictions_suffix
predictions_bucket = predictions_prefix.split('//')[1].split('/')[0]
predictions_prefix = '/'.join(predictions_prefix.split('//')[1].split('/')[1:])
predictions_prefix = f'{predictions_prefix}/{event_date}/'

utils.remove_s3_prefix(state.credentials.cvm_s3, predictions_bucket, predictions_prefix)
assert not utils.list_s3_objects(state.credentials.cvm_s3, predictions_bucket, predictions_prefix), 'S3 predictions prefix was not cleaned'

score_df = pd.DataFrame({
    'model_name': model_version.name,
    'target': target,
    'version_id': model_version.version,
    'model_inference_date': event_date,
    'model_train_date': model_version.tags.get('event_timestamp', train_event_date),
    'customer_id': df['contact_id'].astype('int64'),
    'contact_id': df['contact_id'].astype('int64'),
    'segment': df['segment'] if 'segment' in df.columns else None,
    score: df[score],
})

utils.save_df_to_s3(score_df, state.credentials.cvm_s3, predictions_bucket, predictions_prefix)

objects = utils.list_s3_objects(state.credentials.cvm_s3, predictions_bucket, predictions_prefix)
print('Saved prediction objects:', len(objects))
for obj in objects[:20]:
    print(obj)


### 9. Save Data Statistics


In [ ]:
inference_data_stat_prefix = state.settings.s3_permanent_prefix / inference_data_stat_suffix
inference_data_stat_bucket = inference_data_stat_prefix.split('//')[1].split('/')[0]
inference_data_stat_prefix = '/'.join(inference_data_stat_prefix.split('//')[1].split('/')[1:])
data_stat_prefix = f'{inference_data_stat_prefix}/{event_date}/'

utils.remove_s3_prefix(state.credentials.cvm_s3, inference_data_stat_bucket, data_stat_prefix)
assert not utils.list_s3_objects(state.credentials.cvm_s3, inference_data_stat_bucket, data_stat_prefix), 'S3 data stat prefix was not cleaned'

report = func.get_data_report(df, features + [score])
utils.save_df_to_s3(report, state.credentials.cvm_s3, inference_data_stat_bucket, data_stat_prefix)

display(report.head())

objects = utils.list_s3_objects(state.credentials.cvm_s3, inference_data_stat_bucket, data_stat_prefix)
print('Saved data stat objects:', len(objects))
for obj in objects[:20]:
    print(obj)
